# C3-gradient-descent — Session 1: Loss Surfaces and the Descent Step

*One class session, roughly 85 minutes. Prerequisites: F4-multivar-calculus
(partial derivatives, the gradient and its direction property, sum-of-squares
gradients) and C2-linear-models (the linear model
$\hat y_i = \sum_k X_{ik} w_k + b$ and the MSE loss
$L = \frac{1}{n}\sum_i (\hat y_i - y_i)^2$, evaluated and differentiated at
given parameters).*

**This session:** C2 left one question deliberately open — given the loss and
its gradient, how do you actually *find* good parameters?
This unit's answer is an algorithm: start somewhere, take small downhill
steps, repeat.
Today: loss functions pictured as landscapes, the descent step
$w \leftarrow w - \eta \nabla L(w)$, exactly how fast it homes in on the
bottom of a quadratic bowl — and exactly how it fails when the step size
$\eta$ is too large.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. From Loss Function to Loss Landscape

**Motivation.**
In C2-linear-models the loss $L(w, b)$ was a formula to evaluate: plug in
parameters, get a number measuring mismatch.
To *choose* parameters, promote that formula to a picture: one axis per
parameter, height = loss.
The whole art of fitting becomes geography — find the lowest point.

**One parameter: a curve.**
For a single parameter, $L(w) = (w - 3)^2$ is our running example — a
parabola with its bottom at $w = 3$.
The picture already contains the whole story of this session: at any $w$,
the slope tells you which way is downhill, and the curvature tells you how
fast the slope changes.

**Two parameters: a surface, drawn with contours.**
For the C2 model with one feature, the parameters are $(w, b)$ and
$$L(w, b) = \frac{1}{n} \sum_{i=1}^{n} \big( \underbrace{w x_i + b}_{\hat y_i} - y_i \big)^2 .$$
This is a surface over the $(w, b)$-plane, and we draw it exactly as F4 drew
functions of two inputs: **contour lines**, curves of equal loss.
Below we use the tiny dataset $x = (0, 1, 2, 3)$, $y = (1, 3, 5, 7)$, which
the model can fit *exactly* with $w = 2$, $b = 1$ — so the landscape bottoms
out at height $0$ at exactly that point.

In [ ]:
ws = np.linspace(-1, 7, 200)
plt.figure(figsize=(6, 3.2))
plt.plot(ws, (ws - 3.0) ** 2)
plt.scatter([3.0], [0.0], color="C3", zorder=3, label="minimum at w = 3")
plt.xlabel("w")
plt.ylabel("L(w)")
plt.title(r"$L(w) = (w-3)^2$: a 1-D loss landscape")
plt.legend()
plt.show()

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 3.0, 5.0, 7.0])

w_vals = np.linspace(-1, 5, 121)          # (121,)
b_vals = np.linspace(-2, 4, 121)          # (121,)
# broadcast an (nb, nw, n) prediction block, then average the squared error over the data axis
preds = b_vals[:, None, None] + w_vals[None, :, None] * x[None, None, :]
L_surface = ((preds - y[None, None, :]) ** 2).mean(axis=2)   # (121, 121): rows walk b, columns walk w

plt.figure(figsize=(6, 4.5))
cs = plt.contour(w_vals, b_vals, L_surface, levels=14)
plt.clabel(cs, inline=True, fontsize=7)
plt.scatter([2.0], [1.0], color="C3", zorder=3, label="minimum (w, b) = (2, 1)")
plt.xlabel("w")
plt.ylabel("b")
plt.title("MSE loss surface for the 4-point dataset")
plt.legend()
plt.show()

Read the contour plot like a hiking map, exactly as in F4: each closed curve
is a set of $(w, b)$ pairs with equal loss; curves crowded together mean
steep terrain; the nested rings shrink toward the single lowest point
$(2, 1)$.
Every loss we meet in this unit is such a landscape — most with far more
than two dimensions, where the *picture* fails but the *geometry* (heights,
slopes, downhill directions) carries over unchanged.

### Checkpoint 1

1. For $L(w_1, w_2) = (w_1 - 4)^2 + (w_2 + 2)^2$: where is the minimum,
   and what is the loss value there?
2. By hand, for the dataset $x = (0, 2)$, $y = (1, 5)$: evaluate the MSE
   loss $L(w, b)$ of the model $\hat y_i = w x_i + b$ at $(w, b) = (1, 1)$.
3. On the contour plot above, what does it mean about the surface where
   neighboring contour lines lie close together?

## 2. Which Way Is Down? The Negative Gradient

**Motivation.**
A hiker on the loss surface wants the quickest way down.
F4 proved the tool: at any point, the gradient $\nabla L$ points in the
direction of **steepest increase**, with $\lVert \nabla L \rVert$ the rate of
that increase.
So the direction of **steepest decrease** is exactly the opposite vector,
$-\nabla L$.

**On the bowl $L(w_1, w_2) = w_1^2 + 4 w_2^2$:**
$$\nabla L = (2 w_1,\; 8 w_2), \qquad -\nabla L = (-2 w_1,\; -8 w_2).$$
Note what $-\nabla L$ is *not*: it is not "the arrow pointing at the
minimum."
At $(2, 1)$ the minimum lies in direction $(-2, -1)$, but
$-\nabla L = (-4, -8)$ — much more of the arrow fights the steep $w_2$
direction than the shallow $w_1$ direction.
Steepest descent is a *local* compass, not a map to the bottom.

In [ ]:
gw = np.linspace(-2.5, 2.5, 11)
gb = np.linspace(-1.5, 1.5, 9)
W1 = gw[None, :] + 0 * gb[:, None]        # broadcast to the full (9, 11) grid
W2 = 0 * gw[None, :] + gb[:, None]
L_bowl = W1**2 + 4 * W2**2

plt.figure(figsize=(6, 4.2))
cs = plt.contour(gw, gb, L_bowl, levels=10)
plt.clabel(cs, inline=True, fontsize=7)
plt.quiver(W1, W2, -2 * W1, -8 * W2, color="C0", width=0.003)
plt.scatter([0], [0], color="C3", zorder=3)
plt.xlabel("$w_1$")
plt.ylabel("$w_2$")
plt.title(r"$-\nabla L$ arrows on $L = w_1^2 + 4w_2^2$ (minimum at the red dot)")
plt.show()

Every arrow crosses its contour line at a right angle (F4's direction
property) and leans toward the steep axis — arrows far from the $w_1$-axis
are dominated by their $w_2$ component, because that is where the surface
rises four times faster.

### Checkpoint 2

1. For $L(w_1, w_2) = w_1^2 + 4 w_2^2$: compute $\nabla L$ at $(1, 1)$
   and the direction of steepest decrease there.
2. Where on this bowl is $\nabla L = (0, 0)$, and what is special about
   that point?
3. True or false, with a one-line reason: $-\nabla L$ always points
   straight at the minimum.

## 3. The Descent Step

**Definition.**
**Gradient descent** turns the compass into an algorithm.
Pick a starting point and a **learning rate** $\eta > 0$ (the Greek letter
"eta"), then repeat:
$$w \;\leftarrow\; w - \eta\, \nabla L(w).$$
Each step moves in the locally steepest downhill direction, by an amount
$\eta \lVert \nabla L \rVert$ — big steps where the surface is steep, small
steps where it flattens, and *no* step exactly at a point with
$\nabla L = 0$.
The learning rate is the one knob: it scales how far a single step travels.

**Worked example, by hand (1-D).**
$L(w) = (w - 3)^2$, so $L'(w) = 2(w - 3)$.
Start $w_0 = 0$, take $\eta = 0.1$:
$$w_1 = w_0 - \eta L'(w_0) = 0 - 0.1 \cdot 2 \cdot (0 - 3) = 0.6 .$$

**Worked example, by hand (2-D).**
$L(w_1, w_2) = w_1^2 + 4 w_2^2$ at $(2, 1)$ with $\eta = 0.1$:
$\nabla L = (4, 8)$, so the step lands at
$$(2, 1) - 0.1 \cdot (4, 8) = (1.6,\; 0.2).$$
Both coordinates moved toward $0$ — but $w_2$ covered $80\%$ of its distance
in one step while $w_1$ covered $20\%$, again because the bowl is steeper in
$w_2$.

In code, the step is one line of array arithmetic — and the *repetition* is
a loop.
In this unit, unlike the broadcasting drills of F1–F4, a loop **over descent
steps** is not a style violation: iteration is the algorithm itself.
(Loops over data points or features remain banned in the usual way — each
gradient evaluation stays vectorized.)

In [ ]:
def descent_step(w, grad, eta):
    """One gradient-descent update; w and grad may be scalars or arrays."""
    return w - eta * grad


# the two hand computations, checked:
w1 = descent_step(0.0, 2 * (0.0 - 3.0), eta=0.1)
print("1-D step from w = 0:", round(w1, 10), "   (hand: 0.6)")

w2d = descent_step(np.array([2.0, 1.0]), np.array([4.0, 8.0]), eta=0.1)
print("2-D step from (2, 1):", w2d, "   (hand: [1.6, 0.2])")

### Checkpoint 3

1. $L(w) = (w - 3)^2$, $w_0 = 5$, $\eta = 0.2$: compute $w_1$ by hand.
2. On $L(w_1, w_2) = w_1^2 + 4 w_2^2$, take one step from $(2, 1)$ with
   $\eta = 0.25$. What happened to the $w_2$ coordinate, and why is that a
   warning sign?
3. Complete the sentence: a descent step travels the distance
   $\eta \cdot (\dots)$ — so steps automatically shrink as the iterate
   approaches a minimum because $(\dots)$.

## 4. Convergence on a Quadratic Bowl: the Trace

**Motivation.**
On the quadratic bowl we can predict *everything* about descent with
Calc AB algebra — which makes the bowl the reference case against which all
stranger behavior is measured.

**The contraction.**
For $L(w) = (w - m)^2$ (bottom at $m$), the update is
$$w_{t+1} = w_t - \eta \cdot 2 (w_t - m)
\;\;\Longrightarrow\;\;
w_{t+1} - m = (1 - 2\eta)(w_t - m).$$
Write $e_t = w_t - m$ for the **error**.
Every step multiplies the error by the same **factor $1 - 2\eta$**:
$$e_t = (1 - 2\eta)^t\, e_0 .$$
For $\eta = 0.1$ the factor is $0.8$: each step erases exactly $20\%$ of the
remaining distance.
Watch it happen — the ratio column below prints $0.8000$ in every row:

In [ ]:
m, eta = 3.0, 0.1
w = 0.0
print(" t       w_t     error e_t   ratio e_t / e_(t-1)")
prev = None
for t in range(9):
    e = w - m
    ratio = "      —" if prev is None else f"{e / prev:7.4f}"
    print(f"{t:2d}  {w:8.4f}   {e:9.4f}   {ratio}")
    prev = e
    w = descent_step(w, 2 * (w - m), eta)

Geometric decay is fast: after $t$ steps the error is $0.8^t \cdot (-3)$,
so ~10 steps cut the distance by an order of magnitude, ~21 steps by two.
The same logic runs the 2-D bowl $L = w_1^2 + 4 w_2^2$ **per coordinate**:
$$w_1 \text{-error} \times (1 - 2\eta), \qquad w_2\text{-error} \times (1 - 8\eta),$$
because $\partial L / \partial w_1 = 2 w_1$ and
$\partial L/\partial w_2 = 8 w_2$.
One learning rate, two different contraction speeds — the path below first
collapses onto the shallow valley floor (the fast $w_2$ direction), then
crawls along it:

In [ ]:
eta = 0.1
pt = np.array([2.4, 1.4])
path = [pt.copy()]
for _ in range(30):
    pt = descent_step(pt, np.array([2 * pt[0], 8 * pt[1]]), eta)
    path.append(pt.copy())
path = np.array(path)

plt.figure(figsize=(6, 4.2))
cs = plt.contour(gw, gb, L_bowl, levels=12)
plt.clabel(cs, inline=True, fontsize=7)
plt.plot(path[:, 0], path[:, 1], marker="o", markersize=3, color="C3")
plt.xlabel("$w_1$")
plt.ylabel("$w_2$")
plt.title(f"30 descent steps, eta = {eta}: fast in $w_2$ (factor 0.2), slow in $w_1$ (factor 0.8)")
plt.show()
print("final point:", np.round(path[-1], 6))

### Checkpoint 4

1. On $L(w) = (w - 3)^2$ with $\eta = 0.25$: what is the error factor per
   step, and what is the error after 4 steps starting from $w_0 = 0$?
2. For $L(w) = 5 (w - 1)^2$: derive the error factor per step (differentiate
   first!), and find the $\eta$ that lands **exactly** on the minimum in a
   single step from any start.

## 5. Too Big a Step: Oscillation and Divergence

**Motivation.**
The factor $1 - 2\eta$ does not stay friendly.
As $\eta$ grows past $\tfrac12$, the factor turns negative — each step
*overshoots* the minimum and lands on the other side.
While $|1 - 2\eta| < 1$ that is survivable; once $|1 - 2\eta| > 1$ every
step makes things worse, no matter how close you started.
On $L(w) = (w - 3)^2$ this flip happens at $\eta = 1$.

Two runs below, printed then plotted.
Read them against their factors:

- $\eta = 0.75$: factor $1 - 2\eta = -0.5$.
  The very first step jumps from $0$ to $4.5$ — *past* the minimum at $3$.
  From then on the iterates alternate sides of $3$ while the error halves in
  size each step: $-3.0000,\; 1.5000,\; -0.7500,\; 0.3750, \dots$
  Overshooting, but converging.
- $\eta = 1.1$: factor $-1.2$.
  The iterates again alternate sides, but each error is $1.2\times$ the last:
  $-3.0000,\; 3.6000,\; -4.3200,\; 5.1840, \dots$
  After seven steps the iterate sits at $13.7495$ — more than three times as
  far from the minimum as its starting point.
  Diverging.

In [ ]:
def run_1d(eta, steps=7, w0=0.0, m=3.0):
    w = w0
    ws = [w]
    for _ in range(steps):
        w = descent_step(w, 2 * (w - m), eta)
        ws.append(w)
    return np.array(ws)


for eta in (0.75, 1.1):
    ws = run_1d(eta)
    print(f"eta = {eta}   (factor 1 - 2*eta = {1 - 2 * eta:.1f})")
    print("  w_t: " + "  ".join(f"{v:8.4f}" for v in ws))
    print("  e_t: " + "  ".join(f"{v:8.4f}" for v in ws - 3.0))
    print()

In [ ]:
plt.figure(figsize=(6.5, 3.4))
for eta, style in ((0.1, "-o"), (0.75, "-s"), (1.1, "-^")):
    plt.plot(run_1d(eta), style, markersize=4, label=f"eta = {eta}")
plt.axhline(3.0, color="gray", linewidth=0.8, label="minimum w = 3")
plt.xlabel("step t")
plt.ylabel("$w_t$")
plt.title("Same bowl, three learning rates")
plt.legend()
plt.show()

The plot is the unit's most important picture: the same loss, the same
start, and the learning rate alone decides between smooth convergence
($\eta = 0.1$), damped overshooting ($\eta = 0.75$), and explosion
($\eta = 1.1$).

Two footnotes that generalize beyond this bowl:

- **Steeper bowls tighten the limit.** On $L(w) = a (w - m)^2$ the factor is
  $1 - 2 a \eta$, so divergence starts at $\eta = 1/a$: the larger the
  curvature $a$, the smaller the safe learning rates.
  (You will map such a boundary precisely in p10 and p11.)
- **In several dimensions the steepest direction rules.** On
  $w_1^2 + 10 w_2^2$, the $w_2$ factor $1 - 20\eta$ hits $-1$ first; $\eta$
  must satisfy the *steepest* curvature even if every other direction could
  tolerate far more (p07).

### Checkpoint 5

1. On $L(w) = (w - 3)^2$ with $\eta = 0.9$: compute the error factor.
   Does the run overshoot? Does it converge?
2. Still on $L(w) = (w - 3)^2$: for which exact $\eta$ do the iterates
   neither converge nor diverge, and what do they do instead?
3. A classmate's $\eta = 0.3$ diverges on $L(w) = 5(w - 1)^2$ yet converges
   on $L(w) = (w - 3)^2$. Reconcile this with the two error factors.

## 6. Worked Exam-Style Example 1: Normal-Form Multiple Choice

Descent-step arithmetic is natural exam material: two or three hand
updates, wrapped in the numeric normal form so that exactly one decoded
answer is right.
Here is one in full, in the real register.

---

**Problem (reasoning is not required; no code needed).**
Let $L(w) = (w - 6)^2$.
Starting from $w_0 = 1$, take two gradient-descent steps
$w \leftarrow w - \eta L'(w)$ with $\eta = 0.2$.
The resulting $w_2$ is a positive rational number; write it in lowest terms
as $p/q$ with $\gcd(p, q) = 1$, $q > 0$.
What is $p + q$?

A. 4  B. 21  C. 26  D. 31  E. 52

---

**Solution.**

*Step 1 — set up the error form (faster than stepping literally).*
$L'(w) = 2(w - 6)$, so each step multiplies the error $e_t = w_t - 6$ by
$1 - 2\eta = 0.6$.

*Step 2 — two steps.*
$e_0 = 1 - 6 = -5$, so $e_2 = 0.6^2 \cdot (-5) = -1.8$, giving
$w_2 = 6 - 1.8 = 4.2$.

*Step 3 — decode the normal form.*
$4.2 = 42/10 = 21/5$ in lowest terms, so $p + q = 21 + 5 = 26$:
**answer C**.
The traps are built in: stopping after one step gives $w_1 = 3 = 3/1$, i.e.
$4$ (choice A); forgetting to reduce $42/10$ gives $52$ (choice E).

*Step 4 — the free cross-check.*
Whenever code is allowed at all, replay the two steps numerically; the loop
must print $4.2$.

In [ ]:
w = 1.0
for _ in range(2):
    w = descent_step(w, 2 * (w - 6.0), eta=0.2)
print("w_2 =", round(w, 10), "   -> 21/5, p + q = 26")

### Checkpoint 6

1. Redo the problem with $\eta = 0.5$ (everything else unchanged): what are
   $w_1$ and $w_2$, and what is $p + q$ now?
2. Redo it with $\eta = 1.2$: compute $e_2$. Is $w_2$ closer to or farther
   from the minimum than $w_0$, and what regime from Section 5 is this?

## 7. Common Pitfalls I

**Pitfall 1 — the sign error: `+` instead of `-`.**
The single most common descent bug is
`w = w + eta * grad`.
This is gradient **ascent**: every step moves in the steepest *uphill*
direction, and on a bowl the loss grows by the factor $(1 + 2\eta)^2$ per
step — smoothly, monotonically, no oscillation.
That signature matters: a *sign error* climbs steadily; a *too-large $\eta$*
(Pitfall 2) overshoots and alternates sides.
Watching whether the iterates alternate tells you which bug you have.

In [ ]:
w = 0.0
print("BROKEN (w = w + eta * grad):")
print(" t     w_t      L(w_t)")
for t in range(5):
    print(f"{t:2d}  {w:8.4f}  {(w - 3.0) ** 2:9.4f}")
    w = w + 0.1 * 2 * (w - 3.0)          # BROKEN: ascent, not descent

w = 0.0
for _ in range(5):
    w = w - 0.1 * 2 * (w - 3.0)          # fixed
print(f"fixed after 5 steps: w = {w:.4f}, moving toward 3 as it should")

The broken run marches monotonically *away* from the minimum — $0.0000,
-0.6000, -1.3200, -2.1840, \dots$ — with the loss rising every line
($9.00 \to 12.96 \to 18.66 \to \dots$, each $\times\,1.44 = 1.2^2$).
No oscillation: that is the ascent signature.

**Pitfall 2 — $\eta$ too large or too small.**
Both ends fail, differently.
Too large you saw in Section 5: alternating signs, growing error.
Too small never *breaks* — it just quietly wastes the compute budget:

In [ ]:
w = 0.0
for _ in range(60):
    w = w - 0.001 * 2 * (w - 3.0)
print(f"eta = 0.001: after 60 steps w = {w:.4f}  (only ~11% of the way to 3)")

With $\eta = 0.001$ the factor is $0.998$: sixty steps recover barely $11\%$
of the distance ($w \approx 0.34$ of the way from $0$ toward $3$), and
reaching $1\%$ error would take about $2300$ steps.
Slowness looks like "descent works, loss decreases" — check the *rate*, not
just the direction.

**Pitfall 3 — mean loss vs sum loss: the hidden factor of $n$.**
The pinned C2 loss is the **mean** squared error, $\frac{1}{n}\sum_i(\cdot)^2$.
Drop the $\frac1n$ and every gradient silently becomes $n$ times larger, so a
learning rate tuned for the mean loss can explode on the sum loss.
Demo with $n = 50$ identical points ($x_i = 1$, $y_i = 2$), where
$L_{\text{mean}} = (w-2)^2$ and $L_{\text{sum}} = 50\,(w-2)^2$:

In [ ]:
eta = 0.3
w = 0.0
tr = [w]
for _ in range(3):
    w = w - eta * 2 * (w - 2.0)          # mean-loss gradient
    tr.append(w)
print("mean loss, eta = 0.3   :", [round(v, 3) for v in tr], " (factor 0.4, converging)")

w = 0.0
tr = [w]
for _ in range(3):
    w = w - eta * 100 * (w - 2.0)        # BROKEN: sum-loss gradient (n = 50 -> 100(w-2)), same eta
    tr.append(w)
print("sum  loss, eta = 0.3   :", [round(v, 1) for v in tr], " (factor -29, exploding)")

w = 0.0
tr = [w]
for _ in range(3):
    w = w - (eta / 50) * 100 * (w - 2.0)  # fixed: retune eta by 1/n
    tr.append(w)
print("sum  loss, eta = 0.3/50:", [round(v, 3) for v in tr], " (matches the mean-loss run)")

The same mismatch also breaks *comparisons*: a sum loss of $90$ on
$n = 500$ points and a sum loss of $9$ on $n = 50$ points describe **equally
good** fits.
Mean-MSE values are comparable across dataset sizes and batch sizes; raw
sums are not.
Keep the $\frac1n$ — and if a codebase doesn't, divide $\eta$ by $n$ and
distrust every cross-dataset loss comparison in sight.

### Checkpoint 7

1. A classmate's loss increases smoothly every step, never alternating.
   Another's iterates jump to alternating sides of the minimum with growing
   size. Diagnose each.
2. You switch from mean-MSE to sum-of-squares loss on $n = 200$ points.
   How must $\eta$ change to reproduce the old run exactly?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Minimum at $(w_1, w_2) = (4, -2)$; the loss there is $0$.
2. Predictions $\hat y = (1, 3)$; errors $(0, -2)$; $L = \frac{0 + 4}{2} = 2$.
3. The surface is steep there — the loss changes by one full contour step
   over a short distance in the $(w, b)$-plane.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $\nabla L(1, 1) = (2, 8)$; steepest decrease along $-(2, 8)$
   (unit vector $-(2, 8)/\sqrt{68}$).
2. At $(0, 0)$: both partials vanish. It is the minimum — descent takes no
   step there because the step size is $\eta \lVert \nabla L \rVert = 0$.
3. False. It points in the locally steepest downhill direction, which on
   unequal-curvature bowls (Section 2's picture) is not the direction of the
   minimum.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $L'(5) = 4$, so $w_1 = 5 - 0.2 \cdot 4 = 4.2$.
2. $\nabla L(2, 1) = (4, 8)$; new point $(1, -1)$. The $w_2$ coordinate
   *overshot* through $0$ to the other side — with $1 - 8\eta = -1$, the
   $w_2$ error will bounce between $\pm 1$ forever: $\eta = 0.25$ sits
   exactly on that coordinate's stability boundary.
3. A step travels $\eta \cdot \lVert \nabla L(w) \rVert$; near a minimum the
   gradient shrinks toward $\mathbf{0}$, so the steps shrink automatically.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Factor $1 - 2 \cdot 0.25 = 0.5$; error after 4 steps
   $= 0.5^4 \cdot (-3) = -3/16 = -0.1875$.
2. $L'(w) = 10(w - 1)$, so the factor is $1 - 10\eta$; it is $0$ at
   $\eta = 0.1$ — one step lands exactly on $w = 1$ from any start.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Factor $1 - 1.8 = -0.8$: it overshoots (negative factor) *and* converges
   ($|{-0.8}| < 1$) — the damped-oscillation regime.
2. $\eta = 1$: factor $-1$. The iterates bounce between $w_0$ and $6 - w_0$
   forever, at constant distance from the minimum.
3. The factors are $1 - 10\eta$ and $1 - 2\eta$. At $\eta = 0.3$:
   $1 - 3 = -2$ (diverges) vs $1 - 0.6 = 0.4$ (converges).
   Divergence is a property of $\eta$ *relative to the curvature*, not of
   $\eta$ alone.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Factor $1 - 2 \cdot 0.5 = 0$: $w_1 = 6$ exactly, and $w_2 = 6$ (the
   gradient is $0$ there). $6 = 6/1$, so $p + q = 7$.
2. Factor $-1.4$: $e_2 = (-1.4)^2 \cdot (-5) = -9.8$, so $w_2 = -3.8$ —
   distance $9.8 > 5$: farther than the start. This is the divergent regime
   ($|1 - 2\eta| > 1$).

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Smooth monotone increase = the sign error (gradient ascent).
   Alternating sides with growing size = learning rate past the divergence
   threshold. The presence or absence of oscillation separates the two.
2. Sum-loss gradients are $200\times$ larger, so use $\eta / 200$ to
   reproduce the run step for step.

</details>